# Engine de Monitoreo Transaccional y Mitigación de Fraude Operativo
## Estratega de Datos: Moisés Antonio Marín Bernal

### Resumen Ejecutivo:
Diseño y ejecución de un motor de auditoría de datos orientado a la banca comercial en México. 
El sistema automatiza la detección de anomalías transaccionales en tiempo real, enfocándose 
en la protección de activos mediante la identificación de patrones de fraude por proximidad 
y velocidad de cargo.

### Capacidades Técnicas del Proyecto:
* **Ingeniería de Datos:** Generación de flujos de datos sintéticos con lógica de negocio local (MXN).
* **Auditoría SQL:** Consultas de agregación avanzada para la detección de umbrales críticos de gasto.
* **Gobierno de Datos:** Estructura de repositorio profesional bajo estándares de escalabilidad.

In [1]:
import pandas as pd
import numpy as np

print("Entorno de Fraude configurado con éxito")

Entorno de Fraude configurado con éxito


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta  # ESTA ES LA LÍNEA QUE FALTABA

# 1. Configuración de comercios y ciudades nacionales
comercios_mx = ['OXXO', '7-Eleven', 'Mercado Libre', 'Amazon MX', 'Uber Eats', 'Walmart', 'Gasolinera Pemex', 'Apple Store']
ciudades_mx = ['Toluca', 'CDMX', 'Guadalajara', 'Monterrey', 'Querétaro', 'Puebla', 'Mérida', 'Cancún']

# 2. Generamos 1,000 transacciones con sabor local
np.random.seed(42)
n_rows = 1000
data_mx = {
    'id_transaccion': range(1, n_rows + 1),
    'id_usuario': np.random.randint(100, 150, n_rows),
    'monto_mxn': np.round(np.random.uniform(20, 12000), 2),
    'comercio': np.random.choice(comercios_mx, n_rows),
    'ciudad': np.random.choice(ciudades_mx, n_rows),
    'fecha_hora': [datetime(2026, 4, 1) + timedelta(minutes=np.random.randint(0, 40000)) for _ in range(n_rows)],
    'es_fraude': 0 
}

df_mx = pd.DataFrame(data_mx)

# 3. Inyectamos un fraude típico: "El Tarjetazo en Electrónica"
df_mx.loc[10:12, 'id_usuario'] = 140
df_mx.loc[10:12, 'monto_mxn'] = 11500.00
df_mx.loc[10:12, 'comercio'] = 'Apple Store'
df_mx.loc[10:12, 'ciudad'] = 'CDMX'
df_mx.loc[10:12, 'es_fraude'] = 1

base_time_mx = datetime(2026, 4, 28, 15, 30)
for i in range(3):
    df_mx.loc[10+i, 'fecha_hora'] = base_time_mx + timedelta(minutes=i*2)

print("Dataset Nacional generado con éxito.")
df_mx.iloc[10:15]

Dataset Nacional generado con éxito.


,id_transaccion,id_usuario,monto_mxn,comercio,ciudad,fecha_hora,es_fraude
10,11,140,11500.00,Apple Store,CDMX,2026-04-28 15:30:00,1
11,12,140,11500.00,Apple Store,CDMX,2026-04-28 15:32:00,1
12,13,140,11500.00,Apple Store,CDMX,2026-04-28 15:34:00,1
13,14,139,11205.63,Gasolinera Pemex,Puebla,2026-04-06 18:45:00,0
14,15,123,11205.63,Apple Store,Guadalajara,2026-04-20 18:55:00,0


In [4]:
# Guardamos el dataset nacional en nuestra carpeta de datos
df_mx.to_csv('../1_Data/transacciones_fraude_mx.csv', index=False)

print("¡Archivo guardado con éxito en 1_Data/transacciones_fraude_mx.csv!")

¡Archivo guardado con éxito en 1_Data/transacciones_fraude_mx.csv!


In [5]:
import sqlite3

# 1. Creamos una base de datos temporal (vive en la RAM de tu PC)
conexion = sqlite3.connect(':memory:')

# 2. Convertimos tu tabla de Python a una tabla SQL real
df_mx.to_sql('tabla_fraude', conexion, index=False, if_exists='replace')

print("Base de Datos Interna: LISTA")

Base de Datos Interna: LISTA


In [6]:
# Definimos la regla de negocio: más de 2 compras en el mismo lugar = SOSPECHOSO
query_estrategica = """
SELECT 
    id_usuario, 
    comercio, 
    ciudad, 
    COUNT(*) as numero_operaciones, 
    SUM(monto_mxn) as exposicion_total
FROM tabla_fraude
GROUP BY id_usuario, comercio, ciudad
HAVING numero_operaciones > 2
ORDER BY exposicion_total DESC
"""


alertas_fraude = pd.read_sql(query_estrategica, conexion)

print("ALERTA DE SEGURIDAD BANCARIA - PATRONES DETECTADOS:")
alertas_fraude

ALERTA DE SEGURIDAD BANCARIA - PATRONES DETECTADOS:


,id_usuario,comercio,ciudad,numero_operaciones,exposicion_total
0,121,Uber Eats,Cancún,4,44822.52
1,140,Apple Store,CDMX,3,34500.00
2,108,OXXO,Monterrey,3,33616.89
3,109,7-Eleven,Mérida,3,33616.89
4,118,Uber Eats,Monterrey,3,33616.89
5,128,OXXO,Guadalajara,3,33616.89
6,136,Walmart,Guadalajara,3,33616.89
7,138,OXXO,Querétaro,3,33616.89
8,141,Mercado Libre,Mérida,3,33616.89
9,143,Mercado Libre,Mérida,3,33616.89


In [7]:
import os
print(os.getcwd())

C:\Users\mmari\Fraud_Detection_System_Project\2_Notebooks
